# 数据清洗（clean_v2）

**负责人：** 2 号　**阶段：** 1 数据准备

清洗规则写在 `data/clean_v2/clean_v2.py`（这个数据版本冻结的脚本）。本 notebook 调用它，展示每条规则的依据、数字和图表，最后生成 `data/clean_v2/` 里的文件。

## 任务

- [x] 修复编码，还原核对过的缩写，删除无法恢复的字符并用 `emoji_lost` 标记
- [x] 删除完全重复、内容重复、乱码、非英文和没有正文的推文（全部原始行保存在 `clean_v2_all.csv`）
- [x] hashtag 只去掉 `#`、不拆词；生成 `text_clean` 和 `text_norm`（保留否定词和方向词）
- [x] `group_id`：近似重复和 `text_norm` 相同的推文分到同一组
- [x] 特征列：长度、各类计数、大写比例、hashtag 占比、领域标记
- [ ] 和 4 号确认数据划分按 `group_id` 分组

## 产出

- `data/clean_v2/clean_v2.csv`、`clean_v2_all.csv`、`clean_v2_log.md`（`docs/cleaning_log.md` 保留日志副本）
- `figures/cleaning_funnel.png`、`figures/cleaning_label_by_language.png`

In [ ]:
import sys
from pathlib import Path

# 向上查找仓库根目录（含 src/config.py 的文件夹），在任何子文件夹里打开都能导入 src
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("找不到仓库根目录：请在 clone 下来的仓库里启动 Jupyter（Colab 先 %cd 进仓库）")
sys.path.insert(0, str(ROOT))

from src.config import DATA_ROOT, FIGURES_DIR, LABELS, SEED                 # 路径和常量
from src.data import load_clean, load_raw, load_splits                      # 读取数据和划分
from src.evaluate import evaluate, new_run_id, plot_confusion, save_result  # 统一评分
from src.cache import load_embedding, save_embedding, save_preds            # 嵌入和预测结果的读写

print("仓库根目录:", ROOT)
print("DATA_ROOT:", DATA_ROOT)

## 1. Load the frozen recipe

`clean_v2.py` holds every rule of this data version, so the data, the log and this notebook can never drift apart. `build()` returns all 1,698 raw tweets with a `keep` flag and a `drop_reason`, the step log, and the numbers behind each design decision.

In [ ]:
import importlib.util
import re

import matplotlib.pyplot as plt
import pandas as pd

from src.config import CLEAN_DIR, CLEAN_VERSION, PROJECT_ROOT

pd.set_option("display.max_colwidth", 90)

spec = importlib.util.spec_from_file_location("clean_v2", PROJECT_ROOT / "data" / "clean_v2" / "clean_v2.py")
clean_v2 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(clean_v2)

raw = pd.read_csv(clean_v2.RAW_CSV, encoding="utf-8")
full, log, evidence = clean_v2.build(raw)
kept = full[full["keep"]]
print(f"{len(raw):,} raw tweets -> {len(kept):,} retained")

## 2. Evidence for the `__` rule

The source file replaced the right single quote `’` **and the character after it** with two underscores: `it’s` → `it__`, `don’t` → `don__`, `you’re` → `you__e`. Every occurrence of each candidate pattern was read before a rule was written. `you__e` / `we__e` / `they__e` become *'ve* when the next word is a past participle (*we've witnessed*) and *'re* otherwise (*you're holding*). Anything else after a word (`Biden__ speech` is a possessive, `Red__ #bearmarket` an emoji) cannot be told apart, so it is removed and flagged in `emoji_lost`.

In [ ]:
def contexts(pattern, texts, width=22):
    """Every match of a compiled pattern with some surrounding text."""
    return [s[max(0, m.start() - width): m.end() + width].replace("\n", " ") for s in texts for m in pattern.finditer(s)]


rules = [(name, pattern) for name, pattern, _ in clean_v2.CONTRACTION_RULES] + [("'re/'ve", clean_v2.RE_VE_RULE)]
pd.DataFrame([{"rule": name, "matches": len(hits), **{f"example {i + 1}": h for i, h in enumerate(hits[:3])}}
              for name, pattern in rules for hits in [contexts(pattern, raw["text"])]])

## 3. Steps

In [ ]:
steps = pd.DataFrame(log)
steps[["step", "operation", "rows_affected", "rows_remaining", "details"]]

In [ ]:
steps.loc[steps["example"] != "", ["operation", "example"]]

## 4. Rows removed

Rows are removed only when they duplicate another tweet, cannot be read, are not in English, or contain no words. No tweet is removed for its length or as an outlier. Every removed row stays in `clean_v2_all.csv`.

In [ ]:
removed = full[~full["keep"]]
pd.crosstab(removed["drop_reason"], removed["label_name"], margins=True, margins_name="total")

## 5. Why each decision

**Hashtags are not split.** If the labels are VADER scores, VADER saw `#stockmarketcrash` as one unknown token. Splitting it adds the negative word *crash*, which the labeller never saw:

In [ ]:
pd.Series(evidence["vader"]["agreement"], name="agreement with the given labels").to_frame()

In [ ]:
print(f"raw agrees but text_clean does not: {evidence['vader']['gap_rows']} tweets, "
      f"{evidence['vader']['gap_with_contraction']} of them with a restored contraction")
print("label mix of removed non-English tweets:", evidence["non_english_mix"])
print("label mix of removed tweets without words:", evidence["no_content_mix"])
print("retained tweets whose text_norm keeps these words:", evidence["keep_words"])
print("retained tweets in groups / number of groups:", evidence["groups"])

In [ ]:
# Chart colours: sentiment is a polarity, so a diverging scheme (red - grey - blue) with direct labels
SURFACE, INK, INK_2, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#c3c2b7"
SENTIMENT_COLORS = {"Negative": "#e34948", "Neutral": "#c3c2b7", "Positive": "#2a78d6"}


def language_group(row):
    if row["drop_reason"].startswith("non_english"):
        return "Non-English (removed)"
    if row["keep"]:
        return "Too short to detect (kept)" if row["lang"] == "short" else "English (kept)"
    return None


groups = full.apply(language_group, axis=1)
share = pd.crosstab(groups, full["label_name"], normalize="index")
counts = groups.value_counts()
order = ["English (kept)", "Too short to detect (kept)", "Non-English (removed)"]

fig, ax = plt.subplots(figsize=(7.2, 2.6), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
left = pd.Series(0.0, index=order)
for label in ["Negative", "Neutral", "Positive"]:
    values = share.loc[order, label]
    ax.barh(order, values, left=left, height=0.55, color=SENTIMENT_COLORS[label], edgecolor=SURFACE, linewidth=2, label=label)
    for group, v, l in zip(order, values, left):
        if v >= 0.07:
            ax.text(l + v / 2, group, f"{v:.0%}", ha="center", va="center", fontsize=9, color=INK)
    left += values
ax.set_yticks(range(len(order)), [f"{g}  (n={counts[g]:,})" for g in order], color=INK_2, fontsize=9)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.tick_params(axis="x", colors=INK_2, labelsize=8)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(AXIS)
ax.set_title("Label share by detected language", loc="left", fontsize=11, color=INK, pad=24)
ax.legend(ncol=3, frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0), fontsize=9, labelcolor=INK_2)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "cleaning_label_by_language.png", dpi=200, facecolor=SURFACE)
plt.show()

## 6. Before and after

`text_clean` keeps case and punctuation for transformer models; `text_norm` is the lowercased, lemmatised version for the TF-IDF baseline and FastText.

In [ ]:
kept.sample(6, random_state=SEED)[["text_raw", "text_clean", "text_norm"]]

## 7. Cleaning funnel

In [ ]:
reasons = removed["drop_reason"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(7.2, 2.8), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
ax.barh(reasons.index, reasons.values, height=0.55, color="#2a78d6")
for name, v in reasons.items():
    ax.text(v + 0.8, name, str(v), va="center", fontsize=9, color=INK_2)
ax.tick_params(axis="y", colors=INK_2, labelsize=9)
ax.tick_params(axis="x", colors=INK_2, labelsize=8)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(AXIS)
ax.set_xlabel("tweets removed", color=INK_2, fontsize=9)
ax.set_title(f"Rows removed during cleaning ({len(raw):,} → {len(kept):,} tweets)", loc="left", fontsize=11, color=INK)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "cleaning_funnel.png", dpi=200, facecolor=SURFACE)
plt.show()

## 8. Class balance after cleaning

Removing non-English and word-less tweets removes mostly *Neutral* rows, so the classes become less balanced. Stratified splitting, macro-averaged metrics and class weights handle this.

In [ ]:
balance = pd.DataFrame({"raw": full["label_name"].value_counts(), "retained": kept["label_name"].value_counts()})
balance["retained share"] = (balance["retained"] / balance["retained"].sum()).round(3)
print("largest / smallest class:", round(balance["retained"].max() / balance["retained"].min(), 2))
balance

## 9. Save outputs

`clean_v2.main()` writes `clean_v2.csv`, `clean_v2_all.csv` and `clean_v2_log.md` next to the script; running it again gives byte-identical files. The log is also copied to `docs/cleaning_log.md` for the report.

In [ ]:
clean_v2.main()
log_text = (PROJECT_ROOT / "data" / "clean_v2" / "clean_v2_log.md").read_text(encoding="utf-8")
(PROJECT_ROOT / "docs" / "cleaning_log.md").write_text(log_text, encoding="utf-8", newline="\n")

reloaded = load_clean()  # re-reads data/clean_v2/clean_v2.csv and runs check_clean()
print(reloaded.shape, "| data version:", f"clean_v{CLEAN_VERSION}")
reloaded.head(3)